# SASRec Stage 3 Refine Multi-Task Weight-0.1 BPI2012 Colab Train 06

This notebook tests whether lowering `time_loss_weight` from `1.0` to `0.1`
helps the plain `refine_ml50_do035` multi-task model.

Main comparison groups:
- `refine_single_task`
- `refine_multi_task_w1.0`
- `refine_multi_task_w0.1`

Main comparison metric:
- `full ranking + NDCG@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
MULTITASK_REFINE_W01_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_multitask_w01_ndcg10_v2'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('MULTITASK_REFINE_W01_OUTPUT_DIR:', MULTITASK_REFINE_W01_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2
MULTITASK_REFINE_W01_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_multitask_w01_ndcg10_v2
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"
!mkdir -p "$MULTITASK_REFINE_W01_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


## Prepare Stage 3 processed dataset

This notebook regenerates the Stage 3 dataset into the versioned Drive folder
before training, so the run does not depend on any stale local processed files.


In [8]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!ls "$DATA_DIR"


/content/time-aware-behavior-prediction
[info] moved existing output to backup: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2__backup_20260602_044016
[ok] regenerated Stage 3 processed dataset at: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
[ok] metadata written to: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2/stage3_dataset_metadata.json
{
  "timestamp": 0,
  "delta_prev_seconds": 0,
  "delta_start_seconds": 0,
  "delta_next_seconds": 0
}
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [10]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = ['delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

This experiment keeps the plain `refine_ml50_do035` multi-task model and only changes:
- `time_loss_weight: 1.0 -> 0.1`

Fixed settings:
- backbone: `refine_ml50_do035`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison uses all 3 seeds: `42`, `2024`, `7`


## Check prerequisite runs


In [11]:
from pathlib import Path

baseline_required_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
multitask_required_runs = [
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]

print('=' * 80)
print('Single-task baseline prerequisite runs')
baseline_output_dir = Path(BASELINE_NDCG10_OUTPUT_DIR)
for run_name in baseline_required_runs:
    run_dir = baseline_output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')

print('=' * 80)
print('Existing plain multi-task w1.0 runs')
multitask_output_dir = Path(MULTITASK_BASELINE_OUTPUT_DIR)
for run_name in multitask_required_runs:
    run_dir = multitask_output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Single-task baseline prerequisite runs
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS
Existing plain multi-task w1.0 runs
multitask_refine_ml50_do035_s42 EXISTS
multitask_refine_ml50_do035_s2024 EXISTS
multitask_refine_ml50_do035_s7 EXISTS


## Check planned w0.1 runs


In [12]:
planned_multitask_w01_runs = [
    'multitask_refine_ml50_do035_w01_s42',
    'multitask_refine_ml50_do035_w01_s2024',
    'multitask_refine_ml50_do035_w01_s7',
]

output_dir = Path(MULTITASK_REFINE_W01_OUTPUT_DIR)
print('=' * 80)
print('Stage 3 refine plain multi-task w0.1 runs')
for run_name in planned_multitask_w01_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 refine plain multi-task w0.1 runs
multitask_refine_ml50_do035_w01_s42 OK
multitask_refine_ml50_do035_w01_s2024 OK
multitask_refine_ml50_do035_w01_s7 OK


## Train refine multi-task w0.1 runs


In [13]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_w01_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_multitask_w01_ndcg10_v2/multitask_refine_ml50_do035_w01_s42
epoch=1, loss=0.8439
epoch=2, loss=0.4257
epoch=3, loss=0.3478
epoch=4, loss=0.3039
epoch=5, loss=0.2770
valid [task], Top5Acc: 0.4992, Top10Acc: 0.7318, Acc: 0.0587, MacroF1: 0.0764, TimeMAE: 74940.4286, TimeRMSE: 279158.5744, TimeMedAE: 985.1446
valid [full], NDCG@5: 0.6475, HR@5: 0.7592, NDCG@10: 0.7061, HR@10: 0.9451, MRR: 0.6399
valid [sampled], NDCG@5: 0.5535, HR@5: 0.5570, NDCG@10: 0.5606, HR@10: 0.5795, MRR: 0.5681
test [task], Top5Acc: 0.2832, Top10Acc: 0.4831, Acc: 0.0114, MacroF1: 0.0120, TimeMAE:

In [14]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_w01_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_multitask_w01_ndcg10_v2/multitask_refine_ml50_do035_w01_s2024
epoch=1, loss=0.9643
epoch=2, loss=0.4405
epoch=3, loss=0.3470
epoch=4, loss=0.3056
epoch=5, loss=0.2817
valid [task], Top5Acc: 0.4845, Top10Acc: 0.7355, Acc: 0.0955, MacroF1: 0.1587, TimeMAE: 74656.3246, TimeRMSE: 294301.3492, TimeMedAE: 877.8383
valid [full], NDCG@5: 0.6547, HR@5: 0.7747, NDCG@10: 0.6915, HR@10: 0.8939, MRR: 0.6396
valid [sampled], NDCG@5: 0.5553, HR@5: 0.5569, NDCG@10: 0.5609, HR@10: 0.5746, MRR: 0.5702
test [task], Top5Acc: 0.2818, Top10Acc: 0.4479, Acc: 0.0324, MacroF1: 0.0302, TimeMA

In [15]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_w01_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_REFINE_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_refine_multitask_w01_ndcg10_v2/multitask_refine_ml50_do035_w01_s7
epoch=1, loss=0.8446
epoch=2, loss=0.4319
epoch=3, loss=0.3556
epoch=4, loss=0.3144
epoch=5, loss=0.2864
valid [task], Top5Acc: 0.4907, Top10Acc: 0.7365, Acc: 0.0733, MacroF1: 0.1193, TimeMAE: 75245.7965, TimeRMSE: 291127.2684, TimeMedAE: 840.1147
valid [full], NDCG@5: 0.6530, HR@5: 0.7571, NDCG@10: 0.7126, HR@10: 0.9529, MRR: 0.6459
valid [sampled], NDCG@5: 0.5613, HR@5: 0.5656, NDCG@10: 0.5703, HR@10: 0.5940, MRR: 0.5755
test [task], Top5Acc: 0.3622, Top10Acc: 0.5723, Acc: 0.0587, MacroF1: 0.0366, TimeMAE: 

In [16]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [17]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1600)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [18]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
multitask_w10_runs = [
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]
multitask_w01_runs = [
    'multitask_refine_ml50_do035_w01_s42',
    'multitask_refine_ml50_do035_w01_s2024',
    'multitask_refine_ml50_do035_w01_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
multitask_w10_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)
multitask_w01_df = rebuild_df(MULTITASK_REFINE_W01_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = 'refine_single_task'

multitask_w10_subset = multitask_w10_df[multitask_w10_df['run_name'].isin(multitask_w10_runs)].copy()
multitask_w10_subset['variant'] = 'refine_multi_task_w1.0'

multitask_w01_subset = multitask_w01_df[multitask_w01_df['run_name'].isin(multitask_w01_runs)].copy()
multitask_w01_subset['variant'] = 'refine_multi_task_w0.1'

df_compare = pd.concat(
    [baseline_subset, multitask_w10_subset, multitask_w01_subset],
    ignore_index=True,
)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

display_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric', 'time_loss_weight',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_mrr',
    'best_valid_accuracy', 'best_valid_macro_f1', 'best_valid_top5_accuracy', 'best_valid_top10_accuracy',
    'best_test_accuracy', 'best_test_macro_f1', 'best_test_top5_accuracy', 'best_test_top10_accuracy',
    'best_valid_time_mae', 'best_valid_time_rmse', 'best_valid_time_median_ae',
    'best_test_time_mae', 'best_test_time_rmse', 'best_test_time_median_ae',
]

existing_display_cols = [c for c in display_cols if c in df_compare.columns]
df_compare[existing_display_cols]


,run_name,seed,variant,maxlen,dropout_rate,selection_metric,time_loss_weight,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_mrr,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_mrr
0,multitask_refine_ml50_do035_w01_s7,7,refine_multi_task_w0.1,50,0.35,full_valid_ndcg@10,0.1,0.748742,0.954729,0.687120,0.794083,1.0,0.724267
1,multitask_refine_ml50_do035_w01_s42,42,refine_multi_task_w0.1,50,0.35,full_valid_ndcg@10,0.1,0.726858,0.977823,0.652739,0.857813,1.0,0.810967
2,multitask_refine_ml50_do035_w01_s2024,2024,refine_multi_task_w0.1,50,0.35,full_valid_ndcg@10,0.1,0.745362,0.972714,0.680587,0.785974,1.0,0.719212
3,multitask_refine_ml50_do035_s7,7,refine_multi_task_w1.0,50,0.35,full_valid_ndcg@10,1.0,0.732565,0.956292,0.670971,0.711474,1.0,0.619819
4,multitask_refine_ml50_do035_s42,42,refine_multi_task_w1.0,50,0.35,full_valid_ndcg@10,1.0,0.733665,0.968635,0.668249,0.673060,1.0,0.565251
5,multitask_refine_ml50_do035_s2024,2024,refine_multi_task_w1.0,50,0.35,full_valid_ndcg@10,1.0,0.737569,0.941862,0.680081,0.749001,1.0,0.666433
6,refine_ml50_do035_s7,7,refine_single_task,50,0.35,full_valid_ndcg@10,None,0.728826,0.961737,0.660451,0.872416,1.0,0.831208
7,refine_ml50_do035_s42,42,refine_single_task,50,0.35,full_valid_ndcg@10,None,0.735378,0.977276,0.663510,0.891774,1.0,0.858375
8,refine_ml50_do035_s2024,2024,refine_single_task,50,0.35,full_valid_ndcg@10,None,0.741573,0.992973,0.664353,0.775861,1.0,0.707187


In [19]:
summary_metric_cols = sorted([
    c for c in df_compare.columns
    if c.startswith((
        'best_valid_',
        'best_test_at_best_valid_',
        'last_valid_',
        'last_test_',
    ))
])

summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare


best_test_at_best_valid_full_hr@10      best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_mean_rank           best_test_at_best_valid_full_median_rank      best_test_at_best_valid_full_mrr           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_num_eval_users            best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_mean_rank           best_test_at_best_valid_sampled_median_rank           best_test_at_best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_num_eval_users            best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_median_ae            best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_top1_accuracy           best_test_at_best_valid_task_top5_accuracy           best_valid_full_hr@10           best_valid_full_hr@5           best_valid_full_mean_rank           best_valid_full_median_rank      best_valid_full_mrr           best_valid_full_ndcg@10           best_valid_full_ndcg@5           best_valid_full_num_eval_users            best_valid_sampled_hr@10           best_valid_sampled_hr@5           best_valid_sampled_mean_rank            \
                                                     mean  std                              mean       std                                   mean       std                                     mean  std                             mean       std                                 mean       std                                mean       std                                        mean        std                                  mean       std                                 mean       std                                      mean       std                                        mean       std                                mean       std                                    mean       std                                   mean       std                                           mean        std                                  mean       std                                  mean       std                                  mean          std                                        mean        std                                   mean          std                                        mean       std                                       mean       std                                       mean       std                  mean       std                 mean       std                      mean       std                        mean  std                mean       std                    mean       std                   mean       std                           mean        std                     mean       std                    mean       std                         mean       std   
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Interpretation guide:

- compare `refine_multi_task_w1.0` vs `refine_multi_task_w0.1` first
- use `best_test_at_best_valid_full_ndcg@10` as the main decision metric
- if `w0.1` improves ranking meaningfully, then the loss-balance effect also generalizes beyond `anchor`
